# Full analysis: BDT vs DNN interpretability comparison

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np

from data.loader import load_hls4ml_jets, make_scaled_copies
from models.bdt import train_bdt, evaluate_bdt
from models.dnn import train_dnn, evaluate_dnn, dnn_predict_proba_factory
from explainers.shap_explainer import explain_bdt, explain_dnn, mean_abs_importance as shap_mean_abs
from explainers.lime_explainer import build_lime_explainers, explain_with_lime, mean_abs_importance as lime_mean_abs
from metrics.rank_correlation import kendall_tau_between, agreement_table, rank_table, normalize_to_share, per_class_tau
from plots.comparison_plots import scatter_grid_by_class, grouped_bar_by_class, grouped_bar_overall

## 1. Load data

In [ ]:
d = load_hls4ml_jets()
X_train, X_val, X_test = d["X_train"], d["X_val"], d["X_test"]
y_train, y_val, y_test = d["y_train"], d["y_val"], d["y_test"]
feature_names = d["feature_names"]
le = d["label_encoder"]
class_names = list(le.classes_)

scaler, X_train_sc, X_val_sc, X_test_sc = make_scaled_copies(X_train, X_val, X_test)

## 2. Train both models

In [ ]:
bdt = train_bdt(X_train, y_train, X_val, y_val)
y_pred_bdt, y_prob_bdt, auc_bdt = evaluate_bdt(bdt, X_test, y_test, class_names)

model, device = train_dnn(X_train_sc, y_train, X_val_sc, y_val)
y_pred_dnn, y_prob_dnn, auc_dnn = evaluate_dnn(model, device, X_test_sc, y_test, class_names)

assert np.array_equal(y_test, y_test), "Sanity check: same y_test used for both evaluations."

## 3. SHAP on a shared subsample

In [ ]:
N_SHAP_SAMPLES = 20000
rng = np.random.default_rng(42)
shap_idx = rng.choice(len(X_test), N_SHAP_SAMPLES, replace=False)

X_shap_bdt = X_test[shap_idx]
X_shap_dnn = X_test_sc[shap_idx]
y_shap = y_test[shap_idx]

bdt_shap_arr = explain_bdt(bdt, X_shap_bdt, feature_names, n_classes=len(class_names),
                            save_path="../results/bdt_shap_values.npy")
dnn_shap_arr = explain_dnn(model, device, X_shap_dnn, X_train_sc, feature_names, n_classes=len(class_names),
                            save_path="../results/dnn_shap_values.npy")

bdt_overall_imp = shap_mean_abs(bdt_shap_arr)
dnn_overall_imp = shap_mean_abs(dnn_shap_arr)
tau, p = kendall_tau_between(bdt_overall_imp, dnn_overall_imp)
print(f"BDT vs DNN SHAP overall tau = {tau:.4f} (p = {p:.4f})")
print(agreement_table(bdt_overall_imp, dnn_overall_imp, feature_names, "BDT", "DNN"))

## 4. LIME on a smaller shared subsample

In [ ]:
N_LIME_SAMPLES = 4000
X_lime_bdt = X_shap_bdt[:N_LIME_SAMPLES]
X_lime_dnn = X_shap_dnn[:N_LIME_SAMPLES]

explainer_lime_bdt, explainer_lime_dnn = build_lime_explainers(
    X_train, X_train_sc, feature_names, class_names
)

bdt_lime_arr = explain_with_lime(
    explainer_lime_bdt, X_lime_bdt, bdt.predict_proba,
    n_features=len(feature_names), n_classes=len(class_names),
    save_path="../results/bdt_lime_values.npy",
)
dnn_predict_proba = dnn_predict_proba_factory(model, device)
dnn_lime_arr = explain_with_lime(
    explainer_lime_dnn, X_lime_dnn, dnn_predict_proba,
    n_features=len(feature_names), n_classes=len(class_names),
    save_path="../results/dnn_lime_values.npy",
)

## 5. Cross-method agreement: SHAP vs LIME vs XGBoost-native, per class and overall

In [ ]:
xgb_native_imp = normalize_to_share(bdt.feature_importances_)
bdt_shap_lime_subset = bdt_shap_arr[:N_LIME_SAMPLES]

bdt_shap_overall = normalize_to_share(shap_mean_abs(bdt_shap_lime_subset))
bdt_lime_overall = normalize_to_share(lime_mean_abs(bdt_lime_arr))

df_overall = rank_table(
    {"SHAP": bdt_shap_overall, "LIME": bdt_lime_overall, "XGB_native": xgb_native_imp},
    feature_names,
)
print(df_overall)

grouped_bar_overall(
    {"SHAP": bdt_shap_overall, "LIME": bdt_lime_overall, "XGB_native": xgb_native_imp},
    feature_names, save_path="../results/bdt_shap_lime_xgb_overall.png",
    title="BDT overall: SHAP vs LIME vs XGBoost native",
)

## 6. Per-class SHAP-vs-DNN scatter + summary table

In [ ]:
per_class_results = per_class_tau(bdt_shap_arr, dnn_shap_arr, class_names, shap_mean_abs)
for cls, tau_cls, p_cls in per_class_results:
    print(f"{cls:<10} tau = {tau_cls:.4f}  p = {p_cls:.4f}")

bdt_imp_per_class = np.array([shap_mean_abs(bdt_shap_arr, cls_idx=i) for i in range(len(class_names))])
dnn_imp_per_class = np.array([shap_mean_abs(dnn_shap_arr, cls_idx=i) for i in range(len(class_names))])

scatter_grid_by_class(
    bdt_imp_per_class, dnn_imp_per_class, class_names, feature_names,
    label_a="Mean |SHAP| -- BDT", label_b="Mean |SHAP| -- DNN",
    save_path="../results/shap_bdt_vs_dnn_scatter.png",
)